# Chapter 18: Introducing DisSModel

*Part III — Discrete Spatial Modeling*

## Learning Objectives

By the end of this chapter you will be able to:

- Understand the dual-substrate paradigm (vector and raster)
- Navigate the DisSModel class hierarchy
- Install DisSModel and run Game of Life on both substrates with the same scheduler

In [ ]:
# Standard imports — add chapter-specific imports below
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

Every technique since Chapter 14 has been building a spatial model largely by hand — a raw NumPy grid, a hand-written neighbor loop, a `salabim` event queue assembled one piece at a time. DisSModel exists to stop that from being necessary. It's the Python-native successor to TerraME/LuccME, the modeling framework this book's own research group has developed since 2024 as the 2001-agenda's latest synthesis: open source, interoperable with the scientific Python stack you already know (Chapter 6 onward), and reproducible by construction — a theme Chapter 22 returns to directly. This chapter's job is narrower: introduce the two ideas that make every later chapter in Part III legible — the dual substrate, and the shared class hierarchy every model, whatever its substrate, is built from.

## The Dual-Substrate Idea

DisSModel does not force a choice between the vector world of Chapter 7 and the raster world of Chapter 8 — it makes both first-class, side by side, running the same rule under the same scheduler:

- **Vector substrate** — a model's state lives in a `GeoDataFrame`, exactly like the ones built since Chapter 6. Cells are rows; geometry is exact; a neighborhood is a `libpysal` `W` object, precisely as Chapter 10 built one by hand.
- **Raster substrate** — a model's state lives in NumPy arrays wrapped by a `RasterBackend`, exactly like Chapter 8's elevation grid. Cells are array positions; a neighborhood is index arithmetic — the same `grid[i-1:i+2, j-1:j+2]` slicing Chapter 8 previewed.

The payoff is not "pick whichever you like once and move on" — it's that the *same conceptual model* can be validated on both substrates against each other, the way Part III's coastal case study (Chapter 21) does: implement a process once per substrate, run both on identical input, and report where they agree (and by how much) as a correctness check, not just a performance comparison.

## Class Hierarchy: SpatialModel and RasterModel

Every DisSModel model, regardless of substrate, descends from a common `Model` base class with a four-hook lifecycle:

| Hook | Called | Purpose |
|---|---|---|
| `setup(**kwargs)` | once, right after construction | one-time setup — build a neighborhood, initialize state |
| `pre_execute()` | once per tick, before `execute()` | snapshot state before the transition rule runs |
| `execute()` | once per tick | the transition rule itself — the only required override |
| `post_execute()` | once per tick, after `execute()` | cleanup or logging after the transition rule |

A `Model` registers itself with whichever `Environment` is active the moment it's constructed — no separate `env.add(model)` call, unlike the manual bookkeeping Chapter 16's `salabim` processes needed. `Environment.run()` then drives every registered model tick by tick, calling its three per-tick hooks in order until `end_time` is reached.

`SpatialModel` and `RasterModel` each extend `Model` with exactly the infrastructure their substrate needs, without imposing what the transition rule has to look like:

In [ ]:
from dissmodel.core import Model
from dissmodel.geo import SpatialModel
from dissmodel.geo.raster import RasterModel
from libpysal.weights import Queen

class MySpatialModel(SpatialModel):
    def setup(self):
        # Chapter 10's exact technique, wrapped for you:
        self.create_neighborhood(strategy=Queen)

class MyRasterModel(RasterModel):
    def setup(self):
        # self.backend wraps one or more NumPy arrays, Chapter 8-style
        pass

`SpatialModel.create_neighborhood()` is doing, in one call, exactly what Chapter 10 built by hand: constructing a `libpysal` `W` object over `self.gdf` and caching it on the model — the same caching concern Chapter 10's *Caching Neighborhoods for Performance* section raised, now handled for you.

One layer above `Model` sits `CellularAutomaton`, a `SpatialModel` (or, in its raster form, `RasterModel`) subclass that trades a free-form `execute()` for a stricter contract: implement `rule(idx)`, and `CellularAutomaton.execute()` applies it to every cell automatically. This is the class Chapter 15's hand-rolled Game of Life is about to become a two-line subclass of.

<div class="admonition info">
<p class="admonition-title">Did you know?</p>
<p>DisSModel itself installs from PyPI — <code>pip install dissmodel</code>, the same <code>venv</code> workflow Chapter 5 established. Extension packages like <code>dissmodel-ca</code> (used below) aren't on PyPI yet and install straight from GitHub instead: <code>pip install "git+https://github.com/DisSModel/dissmodel-ca.git"</code>.</p>
</div>

## Game of Life: Vector Substrate

Chapter 15 built Conway's Game of Life from scratch: a grid, a rule, a manual loop over every cell each generation. Here is the same rule as a `CellularAutomaton`, on the vector substrate:

In [ ]:
from dissmodel.core import Environment
from dissmodel.geo import vector_grid
from dissmodel.visualization import Map
from dissmodel_ca.models import GameOfLife
from matplotlib.colors import ListedColormap

gdf = vector_grid(dimension=(20, 20), resolution=1, attrs={"state": 0})

env = Environment(start_time=0, end_time=10)
gol = GameOfLife(gdf=gdf)
gol.initialize()  # seeds the starting alive/dead pattern — setup() only builds the neighborhood

cmap = ListedColormap(["white", "black"])
Map(gdf=gdf, plot_params={"column": "state", "cmap": cmap, "ec": "gray"})

env.run()

`vector_grid()` builds exactly the kind of `GeoDataFrame`-as-simulation-grid Chapter 7 previewed — one row per cell, square polygons, ready for `create_neighborhood()`. `GameOfLife.rule(idx)` reads a cell's own state and its `Queen`-neighborhood's live-neighbor count from `self.gdf`, and returns the next state — Conway's rule, unchanged from Chapter 15, just no longer hand-looped: `CellularAutomaton.execute()` applies `rule` to every row automatically each tick.

## Game of Life: Raster Substrate

The raster version implements the identical rule — same neighbor count, same survive/birth thresholds — on a `RasterModel` instead, trading `rule(idx)` per-cell calls for vectorized NumPy operations over the whole grid at once:

In [ ]:
from dissmodel.geo.raster import raster_grid
from dissmodel.visualization import RasterMap
from dissmodel_ca.models import GameOfLifeRaster

backend = raster_grid(rows=20, cols=20, attrs={"state": 0})

env = Environment(start_time=0, end_time=10)
gol_raster = GameOfLifeRaster(backend=backend)
gol_raster.initialize()

RasterMap(backend=backend, band="state", cmap="Greys", scheme="manual", vmin=0, vmax=1)

env.run()

Internally, `GameOfLifeRaster.execute()` counts each cell's live neighbors with a single convolution-style array operation — the same focal-operation idea Chapter 8 introduced — rather than looking up a per-cell neighborhood object. The rule is mathematically identical; only the mechanism computing "how many live neighbors does this cell have" changes.

## Symmetry at the Call Site

Set the two examples above side by side and the point of this whole chapter becomes visible in four lines:

```python
# Vector
gol = GameOfLife(gdf=gdf)
gol.initialize()
env.run()

# Raster
gol_raster = GameOfLifeRaster(backend=backend)
gol_raster.initialize()
env.run()
```

Same `Environment`, same `.initialize()` call, same `env.run()` — the only difference is which class, and which substrate object, gets constructed. Chapter 17's performance chapter already made the case for *why* a raster substrate can be dramatically faster at scale (vectorized array operations versus a Python-level loop over rows); this symmetry is what makes switching between them, once a model needs that speed, a one-line change rather than a rewrite.

## When to Use Each Substrate

The decision reduces to the same substrate trade-offs Chapters 6 through 13 already established, now applied to a running model instead of a static dataset:

- **Choose vector** when geometry must stay exact — irregular administrative boundaries, a real coastline, parcels that don't tile into a regular grid — or when the model needs attributes a raster cell can't naturally carry (a name, a category, a real area in km²).
- **Choose raster** when the domain already is a regular grid, or when performance at scale matters more than exact geometry — a large cellular automaton, a diffusion process, anything Chapter 17 would flag as a candidate for vectorization.
- **Build both, when correctness matters most.** `brmangue-dissmodel`, the coastal mangrove model Chapter 21 studies in depth, ships vector and raster versions of the *same* equations deliberately, plus a dedicated benchmark comparing their output cell by cell — match percentage, mean absolute error, root-mean-square error. Two independently-implemented substrates agreeing closely is real evidence the *model*, not just one implementation, is correct.

Nothing about this decision is permanent. Chapter 19 builds a complete model from scratch; by Chapter 20 that same discipline is applied to a real land-use change process, on whichever substrate the domain calls for.

## Exercises

1. **Trace the lifecycle.** For the vector Game of Life example, list, in order, every lifecycle hook (`setup`, `pre_execute`, `execute`, `post_execute`) that fires during a single `env.run()` tick. Which one does `GameOfLife` override, and which does `CellularAutomaton` provide for it?
2. **Wrap versus Chapter 15.** Compare `GameOfLife.rule(idx)` to the hand-written neighbor-counting loop you wrote in Chapter 15. What did DisSModel's `create_neighborhood()` + `rule(idx)` contract remove from your code, and what stayed conceptually the same?
3. **Predict the mismatch.** `GameOfLife`'s default neighborhood does not wrap at the grid's edge, unlike some classic Game of Life implementations that treat the grid as a torus. Where would you expect the vector and raster versions to disagree most, if the raster version *did* wrap and the vector version didn't?
4. **Pick a substrate.** For each of the following, name vector or raster and justify it in one sentence: (a) a fire spreading across a 10,000×10,000 cell landscape, (b) a policy model where each cell is a real municipality with a legal boundary, (c) a diffusion process you plan to run thousands of times for a sensitivity analysis (Chapter 25).

In [ ]:
# Your code here

## Summary

### Key concepts introduced

- The dual-substrate idea: vector (`GeoDataFrame`, exact geometry) and raster (`RasterBackend`, vectorized arrays) as first-class, interchangeable substrates for the same model logic
- The shared `Model` lifecycle — `setup → pre_execute → execute → post_execute` — driven automatically by `Environment.run()`, with no manual event registration
- `SpatialModel` and `RasterModel` as substrate-specific infrastructure layers, and `CellularAutomaton`'s `rule(idx)` contract as the class Chapter 15's hand-written Game of Life becomes
- Installing DisSModel (`pip install dissmodel`) and its extension packages (from GitHub, via `pip install "git+..."`) into the `venv` workflow Chapter 5 already established
- The symmetry at the call site — `.initialize()` then `env.run()`, identical regardless of substrate — and why that symmetry is what makes a vector-vs-raster benchmark (Chapter 21) a one-line switch rather than two separate codebases

Chapter 19 builds a complete model of your own from these pieces, start to finish.

## Further Reading

- DisSModel on GitHub: <https://github.com/DisSModel/dissmodel>
- dissmodel-ca on GitHub, including its notebook-based tutorials: <https://github.com/DisSModel/dissmodel-ca>
- libpysal documentation, *Spatial Weights* (the mechanism behind `create_neighborhood()`): <https://pysal.org/libpysal/notebooks/weights.html>